# Sweep 16b MIMIC-IV Full (ICD-9+10) — Modality Ablation × DDI Alpha (Champion Architecture)

**Dataset: MIMIC-IV, ICD-9 + ICD-10 codes (`cohort_tag=mimic4_full`, ~92K patients)**  
**All 2³ = 8 modality contexts × 3 DDI alphas × 5 seeds = 120 runs.**

## How to run (parallel Kaggle accounts)

On **each** account:
1. Run cells **0–4** (install → setup → config → smoke)
2. Run **your assigned context cell(s)** (cells 5–12)
3. Run the matching **export cell** for your context(s)
4. Download the zip

Then **merge locally** and run the Results cell.

---

| Cell | Context | Copy | Notes | Labs | Est. time |
|------|---------|------|-------|------|-----------|
| 5  | naked        | x | x | x | ~15h |
| 6  | copy         | v | x | x | ~15h |
| 7  | notes        | x | v | x | ~15h |
| 8  | labs         | x | x | v | ~15h |
| 9  | notes_copy   | v | v | x | ~15h |
| 10 | labs_copy    | v | x | v | ~15h |
| 11 | notes_labs   | x | v | v | ~15h |
| 12 | full CHAMPION| v | v | v | ~15h |

**Skip-resume:** each cell is safe to restart after a Kaggle timeout.

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn

In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    # Processed files -- anchor on cohort_mimic4_full.pkl
    for find_file in ["cohort_mimic4_full.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    # Code embeddings
    emb_paths = glob.glob("/kaggle/input/**/code_embeddings_mimic4_full.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings_mimic4_full.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    # Note embeddings
    note_paths = glob.glob("/kaggle/input/**/note_embeddings_mimic4_full.pkl", recursive=True)
    if note_paths:
        note_src = note_paths[0]
        note_link = "./data/processed/note_embeddings_mimic4_full.pkl"
        if not os.path.exists(note_link):
            os.symlink(note_src, note_link)
        print(f"Note embeddings linked: {note_src}")
    else:
        print("WARNING: note_embeddings_mimic4_full.pkl not found -- notes contexts will fail")

    # Note global mean (anisotropy correction)
    mean_paths = glob.glob("/kaggle/input/**/note_global_mean_mimic4_full.npy", recursive=True)
    if mean_paths:
        mean_link = "./data/processed/note_global_mean_mimic4_full.npy"
        if not os.path.exists(mean_link):
            os.symlink(mean_paths[0], mean_link)
        print(f"Note global mean linked: {mean_paths[0]}")
    else:
        print("WARNING: note_global_mean_mimic4_full.npy not found")

    # Patch 1: registry.py
    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

    # Patch 2: drug_gnn.py
    gnn_path = "/kaggle/working/src/model/graph_encoders/drug_gnn.py"
    if os.path.exists(gnn_path):
        with open(gnn_path, "r") as rf:
            content = rf.read()
        if "ehr_adj" not in content:
            old = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n    ):"
            new = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n        ehr_adj=None,\n        **kwargs,\n    ):"
            if old in content:
                content = content.replace(old, new)
                with open(gnn_path, "w") as wf:
                    wf.write(content)
                print("  drug_gnn.py patched")

    # Patch 3: dcma_decoder.py
    dcma_path = "/kaggle/working/src/model/decoders/dcma_decoder.py"
    if os.path.exists(dcma_path):
        with open(dcma_path, "r") as rf:
            content = rf.read()
        if "note_proj" not in content:
            content = content.replace(
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)",
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3,\n"
                "                 note_repr_dim: int = 768, lab_repr_dim: int = 400, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)\n"
                "        self.hidden_dim = hidden_dim\n"
                "        self.note_proj = nn.Linear(note_repr_dim, hidden_dim)\n"
                "        self.lab_proj  = nn.Linear(lab_repr_dim,  hidden_dim)",
            )
            content = content.replace(
                "            tokens.append(F.normalize(notes_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.note_proj(F.normalize(notes_repr, dim=-1)).unsqueeze(1))",
            )
            content = content.replace(
                "            tokens.append(F.normalize(labs_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.lab_proj(F.normalize(labs_repr, dim=-1)).unsqueeze(1))",
            )
            with open(dcma_path, "w") as wf:
                wf.write(content)
            print("  dcma_decoder.py patched")
        else:
            print("  dcma_decoder.py already patched")

print("Setup done:", os.getcwd())

In [ ]:
import yaml, subprocess, gc, json, numpy as np
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, training_overrides=None,
                preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if training_overrides:
        for k, v in training_overrides.items():
            cfg["training"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

# Champion architecture (identical to Sweep 16)
CHAMPION_MODEL = {
    "hidden_dim":         128,          # [17D] pure win: +0.0013 Jac, 3x fewer params
    "ar_max_seq_len":     20,
    "note_proj_dim":      64,           # = hidden_dim // 2
    "graph_encoder_type": "drug_gnn",
    "graph_layer_type":   "gcn",
    "hgt_layers":         2,
    "encoder_type":       "imdr_infused",
    "predictor_type":     "heidr",
}
CHAMPION_TRAINING = {
    "bce_weight":           0.3,
    "soft_jaccard_weight":  1.5,
    "margin_weight":        0.05,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
    "--aggregator_type last "
    "--mimic_version 4 "
    "--mimic4_full "
    "--processed_dir data/processed "
    "--embeddings_dir data/embeddings "
)
LAB_FLAGS = "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"

DDI_ALPHAS = [0.0, 0.2, 0.5]

# All 8 modality contexts
CONTEXTS = {
    "naked":      {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"note_method": "none", "lab_dim": 0},
                   "flags":    "",
                   "label":    "[1] naked         (no copy, no notes, no labs)"},
    "copy":       {"model_ov": {"copy_mechanism": True},
                   "prep_ov":  {"note_method": "none", "lab_dim": 0},
                   "flags":    "",
                   "label":    "[2] + copy"},
    "notes":      {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"lab_dim": 0},
                   "flags":    "",
                   "label":    "[3] + notes               (no copy, no labs)"},
    "labs":       {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"note_method": "none", "lab_dim": 400},
                   "flags":    LAB_FLAGS,
                   "label":    "[4] + labs                (no copy, no notes)"},
    "notes_copy": {"model_ov": {"copy_mechanism": True},
                   "prep_ov":  {"lab_dim": 0},
                   "flags":    "",
                   "label":    "[5] + notes + copy        (no labs)"},
    "labs_copy":  {"model_ov": {"copy_mechanism": True},
                   "prep_ov":  {"note_method": "none", "lab_dim": 400},
                   "flags":    LAB_FLAGS,
                   "label":    "[6] + labs + copy         (no notes)"},
    "notes_labs": {"model_ov": {"copy_mechanism": False},
                   "prep_ov":  {"lab_dim": 400},
                   "flags":    LAB_FLAGS,
                   "label":    "[7] + notes + labs        (no copy)"},
    "full":       {"model_ov": {"copy_mechanism": True},
                   "prep_ov":  {"lab_dim": 400},
                   "flags":    LAB_FLAGS,
                   "label":    "[8] + notes + labs + copy  CHAMPION"},
}

SEEDS       = [42, 123, 456, 789, 1024]
reports_dir = Path("experiment_reports/sweep_16b_ablation_mimic4_full")
results_log = []

def alpha_tag(a):
    return str(a).replace(".", "p")

def run_context(ctx_key, ddi_alpha=0.0):
    import torch
    ctx  = CONTEXTS[ctx_key]
    atag = alpha_tag(ddi_alpha)
    cfgp = f"s16bm4f_{ctx_key}_alpha{atag}.yaml"
    make_config(cfgp,
                model_overrides={**CHAMPION_MODEL, **ctx["model_ov"]},
                training_overrides=CHAMPION_TRAINING,
                preprocessing_overrides=ctx["prep_ov"])
    print(f"\n  -- {ctx['label']}  |  ddi_alpha={ddi_alpha} --")
    for seed in SEEDS:
        run_name = f"{ctx_key}_mimic4_full_alpha{atag}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        if list(run_dir.glob("result_*.json")):
            print(f"    [SKIP] {run_name}")
            results_log.append(f"SKIP: {run_name}")
            continue
        log_path = run_dir / "training_log.txt"
        cmd = (f"python -u src/train.py --config {cfgp} "
               f"{BACKBONE_FLAGS} {ctx['flags']} "
               f"--ddi_alpha {ddi_alpha} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"    --- {run_name} ---")
        try:
            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Config ready.  reports_dir =", reports_dir)
print(f"8 contexts x {len(DDI_ALPHAS)} alphas x {len(SEEDS)} seeds = {8*len(DDI_ALPHAS)*len(SEEDS)} total runs")
print()
print("Contexts:", [k for k in CONTEXTS])
print("DDI alphas:", DDI_ALPHAS)

In [ ]:
# Smoke test: 1 epoch x 3 contexts x alpha=0.0
Path("smoke_s16bm4f").mkdir(exist_ok=True)
smoke_cases = ["naked", "notes_labs", "full"]
smoke_results = []
for ctx_name in smoke_cases:
    ctx = CONTEXTS[ctx_name]
    cfg_path = f"s16bm4f_smoke_{ctx_name}.yaml"
    make_config(cfg_path,
                model_overrides={**CHAMPION_MODEL, **ctx["model_ov"]},
                training_overrides=CHAMPION_TRAINING,
                preprocessing_overrides=ctx["prep_ov"],
                smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {ctx['flags']} "
           f"--ddi_alpha 0.0 "
           f"--seed 42 --results_dir smoke_s16bm4f/{ctx_name}")
    print(f"SMOKE [{ctx_name}]: running...")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    for ln in (proc.stdout + proc.stderr).strip().split("\n")[-5:]: print(ln)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {ctx_name}")
    print(f"  >>> {status}\n")
for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("\nAll smoke tests passed.")

In [ ]:
# [1/8] naked -- no copy, no notes, no labs  (~15h total, 3 alphas x 5 seeds)
print("\n" + "="*65)
print("  Context: [1/8] naked         -- no copy, no notes, no labs")
print("  Alphas : " + str(DDI_ALPHAS) + "  Seeds: " + str(SEEDS))
print("="*65)
for alpha in DDI_ALPHAS:
    run_context("naked", ddi_alpha=alpha)
print("\n  Done: naked all alphas complete.")

In [ ]:
# [2/8] copy -- copy only  (~15h total, 3 alphas x 5 seeds)
print("\n" + "="*65)
print("  Context: [2/8] copy          -- copy only")
print("  Alphas : " + str(DDI_ALPHAS) + "  Seeds: " + str(SEEDS))
print("="*65)
for alpha in DDI_ALPHAS:
    run_context("copy", ddi_alpha=alpha)
print("\n  Done: copy all alphas complete.")

In [ ]:
# [3/8] notes -- notes only  (~15h total, 3 alphas x 5 seeds)
print("\n" + "="*65)
print("  Context: [3/8] notes         -- notes only")
print("  Alphas : " + str(DDI_ALPHAS) + "  Seeds: " + str(SEEDS))
print("="*65)
for alpha in DDI_ALPHAS:
    run_context("notes", ddi_alpha=alpha)
print("\n  Done: notes all alphas complete.")

In [ ]:
# [4/8] labs -- labs only  (~15h total, 3 alphas x 5 seeds)
print("\n" + "="*65)
print("  Context: [4/8] labs          -- labs only")
print("  Alphas : " + str(DDI_ALPHAS) + "  Seeds: " + str(SEEDS))
print("="*65)
for alpha in DDI_ALPHAS:
    run_context("labs", ddi_alpha=alpha)
print("\n  Done: labs all alphas complete.")

In [ ]:
# [5/8] notes_copy -- notes + copy, no labs  (~15h total, 3 alphas x 5 seeds)
print("\n" + "="*65)
print("  Context: [5/8] notes_copy    -- notes + copy, no labs")
print("  Alphas : " + str(DDI_ALPHAS) + "  Seeds: " + str(SEEDS))
print("="*65)
for alpha in DDI_ALPHAS:
    run_context("notes_copy", ddi_alpha=alpha)
print("\n  Done: notes_copy all alphas complete.")

In [ ]:
# [6/8] labs_copy -- labs + copy, no notes  (~15h total, 3 alphas x 5 seeds)
print("\n" + "="*65)
print("  Context: [6/8] labs_copy     -- labs + copy, no notes")
print("  Alphas : " + str(DDI_ALPHAS) + "  Seeds: " + str(SEEDS))
print("="*65)
for alpha in DDI_ALPHAS:
    run_context("labs_copy", ddi_alpha=alpha)
print("\n  Done: labs_copy all alphas complete.")

In [ ]:
# [7/8] notes_labs -- notes + labs, no copy  (~15h total, 3 alphas x 5 seeds)
print("\n" + "="*65)
print("  Context: [7/8] notes_labs    -- notes + labs, no copy")
print("  Alphas : " + str(DDI_ALPHAS) + "  Seeds: " + str(SEEDS))
print("="*65)
for alpha in DDI_ALPHAS:
    run_context("notes_labs", ddi_alpha=alpha)
print("\n  Done: notes_labs all alphas complete.")

In [ ]:
# [8/8] full CHAMPION -- notes + labs + copy  (~15h total, 3 alphas x 5 seeds)
print("\n" + "="*65)
print("  Context: [8/8] full CHAMPION -- notes + labs + copy")
print("  Alphas : " + str(DDI_ALPHAS) + "  Seeds: " + str(SEEDS))
print("="*65)
for alpha in DDI_ALPHAS:
    run_context("full", ddi_alpha=alpha)
print("\n  Done: full all alphas complete.")

In [ ]:
total   = len(results_log)
success = sum(1 for r in results_log if "SUCCESS" in r)
failed  = sum(1 for r in results_log if "FAILED"  in r)
crashed = sum(1 for r in results_log if "CRASH"   in r)
skipped = sum(1 for r in results_log if "SKIP"    in r)
print(f"Run log: {total} entries | {success} success | {failed} failed | {crashed} crashed | {skipped} skipped")
if failed + crashed > 0:
    print("\nFailed/crashed:")
    for r in results_log:
        if "SUCCESS" not in r and "SKIP" not in r: print(f"  {r}")

In [ ]:
import json, numpy as np
from pathlib import Path

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

def load_ctx_alpha(ctx_key, ddi_alpha):
    atag = str(ddi_alpha).replace(".", "p")
    runs = []
    for seed in SEEDS:
        for jp in sorted((reports_dir / f"{ctx_key}_mimic4_full_alpha{atag}_seed{seed}").glob("result_*.json")):
            with open(jp) as f: runs.append(json.load(f))
    seen, deduped = set(), []
    for d in runs:
        s = d.get("seed", id(d))
        if s not in seen:
            seen.add(s); deduped.append(d)
    return deduped

def summarize(runs):
    if not runs: return None
    jac   = [get_metric(d, "jaccard")  for d in runs]
    ddi   = [get_metric(d, "DDI Rate") for d in runs]
    f1    = [get_metric(d, "f1")       for d in runs]
    prauc = [get_metric(d, "PRAUC")    for d in runs]
    return dict(jac=np.mean(jac), std=np.std(jac,ddof=1) if len(jac)>1 else 0,
                ddi=np.mean(ddi), f1=np.mean(f1), prauc=np.mean(prauc), n=len(jac))

CTX_ORDER = ["naked","copy","notes","labs","notes_copy","labs_copy","notes_labs","full"]

print("\n" + "="*100)
print("  MIRROR MIMIC-IV Full (ICD-9+10) -- Modality Ablation x DDI Alpha  (Champion Architecture, Sweep 16b)")
print("  Architecture: imdr_infused · gcn · heidr · film · L4_jac_heavy")
print("="*100)

hdr = f"  {'Context':<32}"
for alpha in DDI_ALPHAS:
    hdr += f"  {'a='+str(alpha)+' Jac':>10} {'DDI':>7}"
print(hdr)
print("  " + "-"*95)

for ctx_key in CTX_ORDER:
    ctx   = CONTEXTS[ctx_key]
    label = ctx["label"]
    row   = f"  {label:<32}"
    any_run = False
    for alpha in DDI_ALPHAS:
        s = summarize(load_ctx_alpha(ctx_key, alpha))
        if s is None:
            row += f"  {'(not run)':>10} {'':>7}"
        else:
            any_run = True
            ddi_mark = "v" if s["ddi"] < 0.06 else " "
            row += f"  {s['jac']:>10.4f} {s['ddi']:>6.4f}{ddi_mark}"
    if any_run:
        print(row)
    else:
        print(f"  {label:<32}  (not yet run)")

print("="*100)
print()
print("  v = DDI Rate below 0.06 target")
print()
print(f"  {'Alpha':<10}  {'Best Jac':>9}  {'Best context':>20}  {'Avg DDI (full)':>16}")
print("  " + "-"*65)
for alpha in DDI_ALPHAS:
    best_jac, best_ctx = 0.0, "-"
    for ctx_key in CTX_ORDER:
        s = summarize(load_ctx_alpha(ctx_key, alpha))
        if s and s["jac"] > best_jac:
            best_jac = s["jac"]; best_ctx = ctx_key
    full_s  = summarize(load_ctx_alpha("full", alpha))
    full_ddi = f"{full_s['ddi']:.4f}" if full_s else "n/a"
    print(f"  alpha={alpha:<5}    {best_jac:>9.4f}  {best_ctx:>20}  {full_ddi:>16}")
print()

In [ ]:
# Export: naked
import zipfile; from pathlib import Path
zip_name = "s16bm4f_naked_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.glob("naked_mimic4_full_*/result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.glob("naked_mimic4_full_*/training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")

In [ ]:
# Export: copy
import zipfile; from pathlib import Path
zip_name = "s16bm4f_copy_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.glob("copy_mimic4_full_*/result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.glob("copy_mimic4_full_*/training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")

In [ ]:
# Export: notes
import zipfile; from pathlib import Path
zip_name = "s16bm4f_notes_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.glob("notes_mimic4_full_*/result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.glob("notes_mimic4_full_*/training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")

In [ ]:
# Export: labs
import zipfile; from pathlib import Path
zip_name = "s16bm4f_labs_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.glob("labs_mimic4_full_*/result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.glob("labs_mimic4_full_*/training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")

In [ ]:
# Export: notes_copy
import zipfile; from pathlib import Path
zip_name = "s16bm4f_notes_copy_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.glob("notes_copy_mimic4_full_*/result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.glob("notes_copy_mimic4_full_*/training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")

In [ ]:
# Export: labs_copy
import zipfile; from pathlib import Path
zip_name = "s16bm4f_labs_copy_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.glob("labs_copy_mimic4_full_*/result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.glob("labs_copy_mimic4_full_*/training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")

In [ ]:
# Export: notes_labs
import zipfile; from pathlib import Path
zip_name = "s16bm4f_notes_labs_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.glob("notes_labs_mimic4_full_*/result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.glob("notes_labs_mimic4_full_*/training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")

In [ ]:
# Export: full
import zipfile; from pathlib import Path
zip_name = "s16bm4f_full_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.glob("full_mimic4_full_*/result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.glob("full_mimic4_full_*/training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")

In [ ]:
# Export ALL results
import zipfile; from pathlib import Path
zip_name = "sweep_16b_mimic4_full_ALL_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.rglob("result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.rglob("training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
n = sum(1 for _ in reports_dir.rglob("result_*.json"))
print(f"Exported -> {zip_name}  ({n} result files)")